# 01 — SQL & MongoDB Feeding

Loads `olist_order_payments` → MySQL  
Loads `product_category_name_translation` → MongoDB

> **Credentials** are read from environment variables — never hardcoded.  
> On Databricks use `dbutils.secrets.get(scope, key)` instead of `.env`.

In [ ]:
%pip install mysql-connector-python pymongo python-dotenv

## 1. Configuration

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()  # reads .env when running locally; on Databricks use dbutils.secrets

# ── MySQL ──────────────────────────────────────────────────────────────
MYSQL_HOST = os.getenv('MYSQL_HOST')
MYSQL_PORT = os.getenv('MYSQL_PORT', '3306')
MYSQL_DB   = os.getenv('MYSQL_DB')
MYSQL_USER = os.getenv('MYSQL_USER')
MYSQL_PASS = os.getenv('MYSQL_PASS')

# ── MongoDB ────────────────────────────────────────────────────────────
MONGO_HOST = os.getenv('MONGO_HOST')
MONGO_PORT = os.getenv('MONGO_PORT', '27017')
MONGO_DB   = os.getenv('MONGO_DB')
MONGO_USER = os.getenv('MONGO_USER')
MONGO_PASS = os.getenv('MONGO_PASS')

# ── CSV paths ──────────────────────────────────────────────────────────
ORDER_PAYMENTS_CSV       = '/content/sample_data/olist_order_payments_dataset.csv'
CATEGORY_TRANSLATION_CSV = '/content/sample_data/product_category_name_translation.csv'


## 2. Preview — Order Payments

In [ ]:
import pandas as pd

order_payments = pd.read_csv(ORDER_PAYMENTS_CSV)
order_payments.head()


## 3. Feed MySQL — olist_order_payments

In [ ]:
import mysql.connector
from mysql.connector import Error

table_name = 'olist_order_payments'
connection = None

try:
    connection = mysql.connector.connect(
        host=MYSQL_HOST, database=MYSQL_DB,
        user=MYSQL_USER, password=MYSQL_PASS,
        port=int(MYSQL_PORT)
    )
    if connection.is_connected():
        print('Connected to MySQL', connection.get_server_info())
        cursor = connection.cursor()

        cursor.execute(f'DROP TABLE IF EXISTS {table_name};')
        cursor.execute(f'''
            CREATE TABLE {table_name} (
                order_id             VARCHAR(50),
                payment_sequential   INT,
                payment_type         VARCHAR(20),
                payment_installments INT,
                payment_value        FLOAT
            )''')
        print(f"Table '{table_name}' created")

        data = pd.read_csv(ORDER_PAYMENTS_CSV)
        batch_size, total = 1000, len(data)
        insert_q = f'''
            INSERT INTO {table_name}
              (order_id,payment_sequential,payment_type,payment_installments,payment_value)
            VALUES(%s,%s,%s,%s,%s)'''

        for start in range(0, total, batch_size):
            batch = data.iloc[start:start+batch_size]
            cursor.executemany(insert_q, [tuple(r) for r in batch.itertuples(index=False, name=None)])
            connection.commit()
            print(f'Inserted {start+1}–{min(start+batch_size, total)}')

        print(f'Done — {total} rows inserted into {table_name}')

except Error as e:
    print('MySQL Error:', e)
finally:
    if connection and connection.is_connected():
        cursor.close(); connection.close()
        print('MySQL connection closed')


## 4. Feed MongoDB — product_category_translation

In [ ]:
from pymongo import MongoClient

uri = f'mongodb://{MONGO_USER}:{MONGO_PASS}@{MONGO_HOST}:{MONGO_PORT}/{MONGO_DB}'
client = MongoClient(uri)
db     = client[MONGO_DB]

df      = pd.read_csv(CATEGORY_TRANSLATION_CSV)
records = df.to_dict(orient='records')

collection = db['product_category_translation']
collection.drop()
collection.insert_many(records)
print(f'Inserted {len(records)} documents')
print('Sample:', collection.find_one(projection={'_id': 0}))
client.close()
